# Introducción a RL multiobjetivo

Authores: Juan Diego, Jessica, Daniel Barrero.



La comunidad de MORL (Multiobjective Reinforcement Learning) cuenta con los siguientes recursos:

* MO-Gymnasium: https://mo-gymnasium.farama.org/
  * Una adaptación multiobjetivo de varios ambientes clásicos de Gymnasium

* MORL Baselines: https://lucasalegre.github.io/morl-baselines
  * Una base de algoritmos multiobjetivo que se han desarrollado a lo largo de los años, análoga a Stable-baselines para RL en general.

* Weights and Biases (https://wandb.ai) como herramienta de visualización y evaluación del agente en términos de métricas de aprendizaje.

## 1) **MO-Gymnasium**


### Instalamos MO-Gymnasium con el siguiente comando.

In [1]:
import sys
!pip install mo-gymnasium
!{sys.executable} -m pip install moviepy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 7.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mo-gymnasium] [mo-gymnasium]


### Importamos

In [2]:
import gymnasium as gym
import mo_gymnasium as mo_gym

/opt/miniconda3/envs/cardozoenv/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## Ejemplo de ambiente: Mountain Car



In [3]:
env = mo_gym.make("mo-mountaincar-v0", render_mode="rgb_array")

/opt/miniconda3/envs/cardozoenv/lib/python3.10/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


## Mountain Car:

* El estado tiene dos componentes: (coordenada_x, rapidez). Esto hace que el espacio de estados de Mountain Car sea continuo y 2-dimensional.
* Tres acciones discretas: (0 = acelerar a la izquierda, 1 = no acelerar, 2 = acelerar a la derecha).
* Penalizaciones: -1 por cada paso de tiempo, -1 por acelerar en cualquier dirección.
* Versión multiobjetivo: El vector de recompensas tiene tres componentes. Específicamente, recompensa = [paso_de_tiempo, acelerar_IZQ, acelerar_DER].

## Visualización usando wrappers de Gymnasium

In [ ]:
from gymnasium.wrappers.record_video import RecordVideo

ModuleNotFoundError: No module named 'gymnasium.wrappers.record_video'

In [ ]:
env = RecordVideo(env, "videos/demo", episode_trigger=lambda e: True)

## Ejemplo con una política aleatoria para Mountain Car

El video se guarda como mp4 en la carpeta videos/demo de este notebook.

In [ ]:
env.reset()
done = False

# terminated, truncated
# terminated, info

while not done:
    obs, vec_reward, terminated, info = env.step(env.action_space.sample())
    done = terminated or env.truncated

## 2) Usando algoritmos de MORL-baselines

In [ ]:
!apt-get install libgmp-dev python3-dev
!pip install pycddlib  # required for computing linear support

### Instalamos MORL-baselines con este comando:

In [ ]:
!pip install git+https://github.com/LucasAlegre/morl-baselines.git

### Mountain Car con *Pareto Q-learning* (PQL)

In [ ]:
# Mountain Car

import mo_gymnasium as mo_gym
from mo_gymnasium.wrappers import MORecordEpisodeStatistics

GAMMA = 0.99

env = mo_gym.make("mo-mountaincar-v0", render_mode="rgb_array")
env = MORecordEpisodeStatistics(env, gamma=GAMMA)  # wrapper for recording statistics

eval_env = mo_gym.make("mo-mountaincar-v0", render_mode="rgb_array") # environment used for evaluation

In [ ]:
import numpy as np
from morl_baselines.multi_policy.pareto_q_learning.pql import PQL

agent = PQL(
    env=env,
    ref_point=np.array([0, -50]),  # used to compute hypervolume
    gamma=GAMMA,
    log=True,  # use weights and biases to see the results!
)

agent.train(total_timesteps=100000, eval_env=eval_env, ref_point=np.array([0, -50]))

## Deep Sea Treasure con PQL

El problema de Deep Sea Treasure (DST) (https://mo-gymnasium.farama.org/environments/deep-sea-treasure/) consiste de un submarino que debe buscar tesoros en el fondo del mar.

* La nave percibe una recompensa de -1 por cada paso de tiempo, y obtiene una recompensa positiva según el valor del tesoro que encontró.

* En general, a mayor profundiddad puede encontrar tesoros más valiosos, luego esto describe el balance o *trade-off*.

* El episodio termina cuando el submarino encuentra un tesoro.

In [ ]:
import gymnasium as gym
import mo_gymnasium as mo_gym
from mo_gymnasium.wrappers import MORecordEpisodeStatistics

GAMMA = 0.99

env = mo_gym.make("deep-sea-treasure-v0")
env = MORecordEpisodeStatistics(env, gamma=GAMMA)  # wrapper for recording statistics

eval_env = mo_gym.make("deep-sea-treasure-v0") # environment used for evaluation

In [ ]:
import numpy as np
from morl_baselines.multi_policy.pareto_q_learning.pql import PQL

agent = PQL(
    env=env,
    ref_point=np.array([0, -50]),  # used to compute hypervolume
    gamma=GAMMA,
    log=True,  # use weights and biases to see the results!
)

agent.train(total_timesteps=100000, eval_env=eval_env, ref_point=np.array([0, -50]))

### Visualización del agente de DST

In [ ]:
#from

import morl_baselines.common.utils

def make_gif(env, agent, weight, fullpath: str, fps: int = 50, length: int = 300):
    """Render an episode and save it as a gif."""
    assert "rgb_array" in env.metadata["render_modes"], "Environment does not have rgb_array rendering."

    frames = []
    state, info = env.reset()
    terminated, truncated = False, False
    while not (terminated or truncated) and len(frames) < length:
        frame = env.render()
        frames.append(frame)
        action = agent.select_action(state, agent.score_hypervolume)
        state, reward, terminated, truncated, info = env.step(action)
    env.close()

    from moviepy.editor import ImageSequenceClip

    clip = ImageSequenceClip(list(frames), fps=fps)
    clip.write_gif(fullpath + ".gif", fps=fps)
    print("Saved gif at: " + fullpath + ".gif")


# Create a new environment with render_mode='rgb_array' for video recording
env_dst_render = mo_gym.make("deep-sea-treasure-v0", render_mode='rgb_array')

# Define a sample weight for visualization. You can change this to explore different policies.
sample_weight_dst = np.array([0.5, 0.5]) # Example: Equal preference for both objectives


# Generate the GIF using the wrapped agent
make_gif(env_dst_render, agent, weight=sample_weight_dst, fps=10, fullpath="./deep_sea_treasure_agent")

### Exercise 1:

Solving Resource Gathering using GPI-LS / GPI-PD

- Resource Gathering environment (see https://mo-gymnasium.farama.org/environments/resource-gathering/)
- GPI-LS (see https://lucasalegre.github.io/morl-baselines/algos/multi_policy/mp_mo_q_learning/).

Tips:
- You need to set weight_selection_algo='gpi-ls' and use_gpi_policy=True in the MPMOQLearning constructor in order to use GPI-LS.
- To use GPI-PD (the model-based version of GPI-LS), set dyna=True and gpi_pd=True
- Because this environment has stochastic transitions, set num_eval_episodes_for_front=50 in the train() method in order to evaluate the value of the policies with more precision.
- Use 10 iterations of 10k steps:
 total_timesteps=100000, timesteps_per_iteration=10000
- Use epsilon-greedy exploration with
    initial_epsilon=1.0,
    final_epsilon=0.05,
    epsilon_decay_steps=100000
- Observe the metrics in "eval/" panel of weights and biases (e.g., "eval/eum" for expected utility)

In [ ]:
import gymnasium as gym
import mo_gymnasium as mo_gym
from mo_gymnasium.utils import MORecordEpisodeStatistics
import numpy as np

GAMMA = 0.9
ref_point = np.array([-1., -1., -2.])

env = mo_gym.make("resource-gathering-v0")
env = MORecordEpisodeStatistics(env, gamma=GAMMA)  # wrapper for recording statistics

eval_env = mo_gym.make("resource-gathering-v0") # environment used for evaluation

env.pareto_front(GAMMA) # known Pareto front

In [ ]:
from morl_baselines.multi_policy.multi_policy_moqlearning.mp_mo_q_learning import MPMOQLearning

# Your code here:
agent = MPMOQLearning(
    env,
    initial_epsilon=1.0,
    final_epsilon=0.05,
    epsilon_decay_steps=100000,
    gamma=GAMMA,
    dyna=True,
    gpi_pd=True,
    weight_selection_algo='gpi-ls',
    use_gpi_policy=True
)

agent.train(total_timesteps=100000, timesteps_per_iteration=10000, eval_env=eval_env, num_eval_episodes_for_front=50, ref_point=ref_point)

In [ ]:
env.pareto_front(0.9)

In [ ]:
agent.linear_support.ccs

### Exercise 2:

Use your learned agent and visualize how the learned behaviours change depending on the utility!

Use the make_gif function of morl-baselines (https://lucasalegre.github.io/morl-baselines/features/misc/#morl_baselines.common.utils.make_gif).

How does the policy for the following linear weights differ?
* [0.9, 0.1, 0.0]
* [0.3, 0.7, 0.0]
* [0.0, 1.0, 0.0]

In [ ]:
from morl_baselines.common.utils import make_gif

env2 = mo_gym.make("resource-gathering-v0", render_mode='rgb_array')  # you need to set the render_mode to render the gifs

# Your code here:
make_gif(env2, agent, weight=np.array([0.9, 0.1, 0.0]), fps=10, fullpath="./myagent1")
make_gif(env2, agent, weight=np.array([0.3, 0.7, 0.0]), fps=10, fullpath="./myagent2")
make_gif(env2, agent, weight=np.array([0.0, 1.0, 0.0]), fps=10, fullpath="./myagent3")

## Minecart

Next, we will play with continuous states and function approximation!

In [ ]:
import gymnasium as gym
import mo_gymnasium as mo_gym
from mo_gymnasium.utils import MORecordEpisodeStatistics

### Pareto Conditioned Network (PCN)

Let's solve the Minecart problem (https://mo-gymnasium.farama.org/environments/minecart-deterministic) using PCN (https://lucasalegre.github.io/morl-baselines/algos/multi_policy/pcn)!

In [ ]:
from morl_baselines.multi_policy.pcn.pcn import PCN


GAMMA = 1.0

env = mo_gym.make("minecart-deterministic-v0")
env = MORecordEpisodeStatistics(env, gamma=GAMMA)  # wrapper for recording statistics

eval_env = mo_gym.make("minecart-deterministic-v0") # environment used for evaluation

agent = PCN(
    env,
    scaling_factor=np.array([1.0, 1.0, 0.1, 0.1]),
)

agent.train(1000000,
            eval_env=eval_env,
            ref_point=np.array([-1,-1,-200]),
            max_return=np.array([1.5,1.5,0.0]),
            max_buffer_size=200,
 )

### GPI-Linear Support (GPI-LS)

### Exercise 3

Now try to solve the stochastic version with GPI-LS (https://lucasalegre.github.io/morl-baselines/algos/multi_policy/gpi_pd)!

In [ ]:
from morl_baselines.multi_policy.gpi_pd.gpi_pd import GPILS

GAMMA = 0.98

env = mo_gym.make("minecart-v0")
env = MORecordEpisodeStatistics(env, gamma=GAMMA)  # wrapper for recording statistics

eval_env = mo_gym.make("minecart-v0") # environment used for evaluation

# Your code here:
agent = GPILS(
    env,
    per=True,
    initial_epsilon=1.0,
    final_epsilon=0.05,
    epsilon_decay_steps=200000,
    target_net_update_freq=200,
    gradient_updates=10
)

agent.train(total_timesteps=200000,
            eval_env=eval_env,
            ref_point=np.array([-1,-1,-200])
)